# KDU 2026 AI — Model Optimization (LoRA + RFT)
## All-Phases Google Colab Notebook

| Phase | Topic |
|-------|-------|
| 1 | Dataset preparation — 50 JSONL examples in Llama 3 format + `<|eot_id|>` analysis |
| 2 | Programmable JSON grader — RFT reward signal (PPO / DPO explanation) |
| 3 | LoRA fine-tuning with Unsloth — rank trade-offs, merge, GGUF export for Ollama |

---
**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (free tier, ~15 GB VRAM)
2. Run cells top-to-bottom. Training (~60 steps) takes about 3–5 minutes on T4.
3. Mount Google Drive in the last section to persist all outputs.


---
## 0. Environment Setup


In [4]:
# Verify GPU availability
import subprocess
res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
    print("GPU ready.")
else:
    print("No GPU detected!")
    print("Fix: Runtime -> Change runtime type -> T4 GPU, then re-run all cells.")

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [5]:
%%capture
# Install Unsloth + training stack
# Unsloth auto-detects CUDA version and installs the right build
!pip install unsloth
!pip install datasets trl peft accelerate bitsandbytes transformers sentencepiece


In [6]:
import unsloth, transformers, trl, datasets
print(f"unsloth     {unsloth.__version__}")
print(f"transformers {transformers.__version__}")
print(f"trl          {trl.__version__}")
print(f"datasets     {datasets.__version__}")
print("All dependencies OK.")


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [7]:
import json
import os
from pathlib import Path

# Output directories
OUTPUT_DIR    = Path("outputs/lora_adapter")
MERGED_DIR    = Path("outputs/merged_model")
GGUF_DIR      = Path("outputs/gguf_model")
DATA_PATH     = Path("data/company_syntax_train.jsonl")

for d in [OUTPUT_DIR, MERGED_DIR, GGUF_DIR, DATA_PATH.parent]:
    d.mkdir(parents=True, exist_ok=True)

print("Directory structure ready.")


Directory structure ready.


---
## Phase 1 — Dataset Preparation

**Objective:** Generate 50 JSONL training examples that map company-specific acronyms
(HDO, INC, RCA, CAPA, CRQ, CAB, IAM, UAT, DR, KBA) to a structured JSON output schema.

Each example uses the **Llama 3 instruction format** with explicit `<|eot_id|>` end-of-turn tokens.


In [8]:
# ── Phase 1: Constants ────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are the company syntax normalizer. Return strict JSON only using the "
    "approved schema keys. Do not add markdown, prose, or comments."
)

# Llama 3 special tokens
LLAMA3_BEGIN_OF_TEXT = "<|begin_of_text|>"
LLAMA3_START_HEADER  = "<|start_header_id|>"
LLAMA3_END_HEADER    = "<|end_header_id|>"
LLAMA3_END_OF_TURN   = "<|eot_id|>"

REQUEST_TYPES = [
    {"code": "HDO",  "expansion": "handover",                            "request_type": "operations_handover"},
    {"code": "INC",  "expansion": "incident",                            "request_type": "service_incident"},
    {"code": "RCA",  "expansion": "root_cause_analysis",                 "request_type": "post_incident_review"},
    {"code": "CAPA", "expansion": "corrective_action_preventive_action", "request_type": "corrective_action_plan"},
    {"code": "CRQ",  "expansion": "change_request",                      "request_type": "planned_change"},
    {"code": "CAB",  "expansion": "change_advisory_board_review",        "request_type": "change_review"},
    {"code": "IAM",  "expansion": "identity_and_access_management",      "request_type": "access_request"},
    {"code": "UAT",  "expansion": "user_acceptance_testing",             "request_type": "release_validation"},
    {"code": "DR",   "expansion": "disaster_recovery",                   "request_type": "recovery_readiness"},
    {"code": "KBA",  "expansion": "knowledge_base_article",              "request_type": "knowledge_publish"},
]

DEPARTMENTS = [
    {"code": "OPS", "name": "Operations"},
    {"code": "NET", "name": "Network Engineering"},
    {"code": "SEC", "name": "Security Operations"},
    {"code": "FIN", "name": "Finance Systems"},
    {"code": "HR",  "name": "Human Resources Systems"},
]

PRIORITY_POOL     = ["P1", "P2", "P3", "P2", "P1"]
PROMPT_TEMPLATES  = [
    "Convert this {priority} {code} request for team {dept} into the company JSON schema.",
    "Normalize acronym {code} for department {dept} with {priority} priority and return strict JSON only.",
    "Format the internal {dept} {code} workflow as the approved response object. Severity is {priority}.",
    "Translate company shorthand {code} for team {dept}. Use the standard schema and mark priority as {priority}.",
    "Create the final machine-readable company record for {dept} / {code} / {priority}. Return JSON only.",
]

print(f"Request types: {len(REQUEST_TYPES)} | Departments: {len(DEPARTMENTS)} | Total examples: {len(REQUEST_TYPES)*len(DEPARTMENTS)}")


Request types: 10 | Departments: 5 | Total examples: 50


In [9]:
# ── Phase 1: Helper functions ─────────────────────────────────────────────

def render_llama3_text(messages: list) -> str:
    """Render a message list in Llama 3 format with explicit <|eot_id|> per turn."""
    parts = [LLAMA3_BEGIN_OF_TEXT]
    for msg in messages:
        parts.append(
            f"{LLAMA3_START_HEADER}{msg['role']}{LLAMA3_END_HEADER}\n\n"
            f"{msg['content']}{LLAMA3_END_OF_TURN}"
        )
    return "".join(parts)


def build_examples() -> list:
    """Build 50 dataset examples (10 request types x 5 departments)."""
    examples = []
    for idx, (req, dept) in enumerate(
        (r, d) for r in REQUEST_TYPES for d in DEPARTMENTS
    ):
        req_i    = idx // len(DEPARTMENTS)
        dept_i   = idx % len(DEPARTMENTS)
        priority = PRIORITY_POOL[(req_i + dept_i) % len(PRIORITY_POOL)]
        template = PROMPT_TEMPLATES[idx % len(PROMPT_TEMPLATES)]

        payload = {
            "request_code":      req["code"],
            "request_expansion": req["expansion"],
            "request_type":      req["request_type"],
            "department":        dept["code"],
            "department_name":   dept["name"],
            "priority":          priority,
        }
        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": template.format(
                priority=priority, code=req["code"], dept=dept["code"])},
            {"role": "assistant", "content": json.dumps(payload, separators=(",", ":"))},
        ]
        examples.append({"messages": messages, "text": render_llama3_text(messages)})
    return examples


def validate_examples(examples: list) -> None:
    """Assert count, role order, valid JSON, and correct <|eot_id|> count per example."""
    assert len(examples) == 50, f"Expected 50 examples, got {len(examples)}"
    for i, ex in enumerate(examples, 1):
        roles = [m["role"] for m in ex["messages"]]
        assert roles == ["system", "user", "assistant"], f"Row {i}: bad role order {roles}"
        json.loads(ex["messages"][-1]["content"])     # raises ValueError if not valid JSON
        text = ex["text"]
        assert text.startswith(LLAMA3_BEGIN_OF_TEXT),   f"Row {i}: missing begin_of_text"
        eot_count = text.count(LLAMA3_END_OF_TURN)
        assert eot_count == 3, f"Row {i}: expected 3 <|eot_id|> tokens, got {eot_count}"
        assert text.endswith(LLAMA3_END_OF_TURN),       f"Row {i}: must end with <|eot_id|>"
    print(f"Validation passed: all {len(examples)} examples OK.")


print("Functions defined.")


Functions defined.


In [10]:
# ── Phase 1: Generate, validate, and save ────────────────────────────────

examples = build_examples()
validate_examples(examples)

with DATA_PATH.open("w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=True) + "\n")

print(f"Written: {DATA_PATH} ({DATA_PATH.stat().st_size} bytes)")
print(f"\nSample assistant output (row 1):")
print(examples[0]["messages"][-1]["content"])
print(f"\nSample assistant output (row 6):")
print(examples[5]["messages"][-1]["content"])


Validation passed: all 50 examples OK.
Written: data\company_syntax_train.jsonl (57650 bytes)

Sample assistant output (row 1):
{"request_code":"HDO","request_expansion":"handover","request_type":"operations_handover","department":"OPS","department_name":"Operations","priority":"P1"}

Sample assistant output (row 6):
{"request_code":"INC","request_expansion":"incident","request_type":"service_incident","department":"OPS","department_name":"Operations","priority":"P2"}


In [11]:
# ── Phase 1: Preview with HuggingFace datasets ───────────────────────────

from datasets import load_dataset

ds = load_dataset("json", data_files=str(DATA_PATH), split="train")
print(f"HuggingFace dataset: {len(ds)} rows  |  columns: {ds.column_names}")
print("\n--- Full training text for row 0 ---")
print(ds[0]["text"])
print("\n--- Full training text for row 12 ---")
print(ds[12]["text"])


c:\Users\Dell\Desktop\Phase3-AI\KDU-2026-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: The pyarrow installation is not built with support for the Parquet file format (DLL load failed while importing _parquet: An Application Control policy has blocked this file.)

---
### Phase 1 Q&A — The `<|eot_id|>` Formatting Trap

#### The Attack — What catastrophic behaviour occurs if `<|eot_id|>` is missing?

Without end-of-turn tokens the model **never learns where one conversation turn ends and the next begins**.  
At inference time this produces:

| Symptom | Root cause |
|---------|------------|
| Infinite / runaway generation | No learned stop signal |
| Merged responses | System / user / assistant turns bleed together |
| Broken JSON | Output continues past the closing `}` |
| Hallucinated follow-up turns | Model invents a new user question after answering |

#### The Defense — Llama 3 Instruction Format

Each turn is wrapped with explicit special tokens so the model learns hard boundaries:

```
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>

{system prompt}<|eot_id|>
<|start_header_id|>user<|end_header_id|>

{user message}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>

{JSON output}<|eot_id|>
```

Every training example has **exactly 3 `<|eot_id|>` tokens** — one after system, user, and assistant.  
The validation function above asserts this for all 50 rows before saving.

At inference, the model stops generating as soon as it predicts `<|eot_id|>`,  
so the JSON output is always cleanly terminated.


---
## Phase 2 — Programmable JSON Grader (RFT Reward Signal)

**Objective:** Implement a binary grader that scores model outputs.  
This score becomes the reward signal in Reinforcement Fine-Tuning (RFT).


In [ ]:
# ── Phase 2: Grader function ──────────────────────────────────────────────

def grade_json_output(model_output: str) -> float:
    """
    Binary JSON validity grader.

    Returns:
        1.0  — output is valid JSON  (reward: reinforce this behaviour)
       -1.0  — output is invalid JSON (penalty: discourage this behaviour)
    """
    try:
        json.loads(model_output)
        return 1.0
    except json.JSONDecodeError:
        return -1.0


print("grade_json_output defined.")


In [ ]:
# ── Phase 2: Test cases ───────────────────────────────────────────────────

test_cases = [
    ('{"request_code":"HDO","request_type":"operations_handover"}',    "valid JSON object"),
    ('{"request_code":"INC","priority":"P1","department":"SEC"}',       "valid multi-key JSON"),
    ('{"request_code":"RCA","priority":P2}',                            "missing quotes on value (invalid)"),
    ('{"broken": }',                                                    "syntax error (invalid)"),
    ('not json at all',                                                  "plain text (invalid)"),
    ('',                                                                 "empty string (invalid)"),
    ('null',                                                             "JSON null (valid!)"),
    ('["a", "b"]',                                                       "JSON array (valid)"),
    ('{"request_code":"HDO"}  extra text after',                         "trailing garbage (invalid)"),
]

print(f"{'Score':>6}  {'Label':5}  Description")
print("-" * 55)
for text, desc in test_cases:
    score = grade_json_output(text)
    label = "PASS" if score == 1.0 else "FAIL"
    print(f"{score:+6.1f}  {label:5}  {desc}")


---
### Phase 2 Q&A — How is the reward used in PPO and DPO?

#### PPO (Proximal Policy Optimization)

The grader score is used **directly as a scalar reward** during the PPO training loop:

1. The model generates an output from a prompt.
2. `grade_json_output(output)` is called → returns `+1.0` or `−1.0`.
3. PPO computes the advantage: `A = reward − baseline`.
4. The policy is updated to **increase the probability of outputs with A > 0**  
   and **decrease the probability of outputs with A < 0**.
5. The KL-divergence penalty keeps the model from drifting too far from the base.

Result: over many steps, the model learns to output valid JSON to maximise reward.

#### DPO (Direct Preference Optimization)

DPO uses the grader to **build preference pairs** offline:

1. For the same prompt, collect two model outputs: one valid JSON, one invalid.
2. The valid output becomes the **chosen** response (`y_w`).
3. The invalid output becomes the **rejected** response (`y_l`).
4. DPO fine-tunes the model using the Bradley-Terry loss so that  
   `log P(y_w | x) − log P(y_l | x)` is maximised.

Result: the model is trained to prefer the format-correct output without needing an explicit reward model or PPO rollout loop — making DPO cheaper but less flexible than PPO.

| Dimension | PPO | DPO |
|-----------|-----|-----|
| Reward used | Online (each rollout) | Offline (pre-built pairs) |
| Compute | Higher (RL loop) | Lower (supervised-style) |
| Flexibility | High (any scalar reward) | Requires paired data |
| Stability | Needs careful KL tuning | More stable training |


---
## Phase 3 — LoRA Fine-Tuning with Unsloth

**Model:** `unsloth/Llama-3.2-1B-Instruct-bnb-4bit`  
Small enough for T4 free tier, representative of the Llama 3 architecture.

**Pipeline:**
1. Load base model in 4-bit with Unsloth
2. Attach LoRA adapters (rank 8)
3. Fine-tune with SFTTrainer (60 steps)
4. Test inference + grade output
5. Analyse rank trade-offs (r=8 vs r=16)
6. Merge adapter into base model
7. Export as GGUF (q4_k_m) for Ollama


In [ ]:
# ── Phase 3: Training configuration ──────────────────────────────────────

MODEL_NAME                  = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH              = 2048
LORA_RANK                   = 8        # change to 16 to see trade-off
LORA_ALPHA                  = LORA_RANK
BATCH_SIZE                  = 2
GRADIENT_ACCUMULATION_STEPS = 4        # effective batch = 8
MAX_STEPS                   = 60
LEARNING_RATE               = 2e-4

print(f"Model:            {MODEL_NAME}")
print(f"LoRA rank:        {LORA_RANK}")
print(f"LoRA alpha:       {LORA_ALPHA}")
print(f"Max seq length:   {MAX_SEQ_LENGTH}")
print(f"Batch size:       {BATCH_SIZE}")
print(f"Grad accum steps: {GRADIENT_ACCUMULATION_STEPS}  (effective batch = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"Max steps:        {MAX_STEPS}")
print(f"Learning rate:    {LEARNING_RATE}")


In [ ]:
# ── Phase 3: Load base model in 4-bit ────────────────────────────────────
# Unsloth loads the model with BnB 4-bit quantization.
# This halves VRAM usage vs FP16 with negligible quality loss for fine-tuning.

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = None,       # auto-detect BF16 or FP16
    load_in_4bit  = True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {MODEL_NAME}")
print(f"Vocab size:   {tokenizer.vocab_size:,}")


In [ ]:
# ── Phase 3: Attach LoRA adapters ─────────────────────────────────────────
# Target all attention and feed-forward projection layers.
# Only these low-rank matrices are trained; the rest of the model is frozen.

model = FastLanguageModel.get_peft_model(
    model,
    r               = LORA_RANK,
    target_modules  = [
        "q_proj", "k_proj", "v_proj", "o_proj",      # attention projections
        "gate_proj", "up_proj", "down_proj",           # feed-forward (SwiGLU)
    ],
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = 0,
    bias            = "none",
    use_gradient_checkpointing = "unsloth",
    random_state    = 3407,
    max_seq_length  = MAX_SEQ_LENGTH,
    use_rslora      = False,
    loftq_config    = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,}  ({100*trainable/total:.2f}% of {total:,} total)")


In [ ]:
# ── Phase 3: Load dataset ─────────────────────────────────────────────────

from datasets import load_dataset

train_dataset = load_dataset("json", data_files=str(DATA_PATH), split="train")
print(f"Training rows:  {len(train_dataset)}")
print(f"Columns:        {train_dataset.column_names}")
print(f"\nFirst 300 chars of training text for row 0:")
print(train_dataset[0]["text"][:300])


In [ ]:
# ── Phase 3: Build SFT trainer ────────────────────────────────────────────

from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    per_device_train_batch_size     = BATCH_SIZE,
    gradient_accumulation_steps     = GRADIENT_ACCUMULATION_STEPS,
    warmup_steps                    = 5,
    max_steps                       = MAX_STEPS,
    learning_rate                   = LEARNING_RATE,
    fp16                            = not is_bfloat16_supported(),
    bf16                            = is_bfloat16_supported(),
    logging_steps                   = 5,
    optim                           = "adamw_8bit",
    weight_decay                    = 0.01,
    lr_scheduler_type               = "linear",
    seed                            = 3407,
    output_dir                      = str(OUTPUT_DIR),
    report_to                       = "none",
    save_strategy                   = "steps",
    save_steps                      = MAX_STEPS,
)

trainer = SFTTrainer(
    model               = model,
    tokenizer           = tokenizer,
    train_dataset       = train_dataset,
    dataset_text_field  = "text",
    max_seq_length      = MAX_SEQ_LENGTH,
    dataset_num_proc    = 1,
    packing             = False,
    args                = training_args,
)

print("Trainer ready.")
print(f"Mixed precision: {'BF16' if is_bfloat16_supported() else 'FP16'}")


In [ ]:
# ── Phase 3: Train ────────────────────────────────────────────────────────
# ~60 steps on T4 takes 3-5 minutes.
# Watch the 'loss' column — it should decrease from ~2.x toward ~0.x.

import time
t0 = time.time()

trainer_stats = trainer.train()

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Final loss: {trainer_stats.training_loss:.4f}")


In [ ]:
# ── Phase 3: Save LoRA adapter ────────────────────────────────────────────
# Save only the small adapter weights (~10-50 MB), not the full 1B model.

model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

adapter_files = list(OUTPUT_DIR.glob("*"))
print(f"LoRA adapter saved to: {OUTPUT_DIR}")
for f in adapter_files:
    size = f.stat().st_size / 1024
    print(f"  {f.name:40s}  {size:8.1f} KB")


In [ ]:
# ── Phase 3: Test inference ───────────────────────────────────────────────
# Switch model to fast inference mode (fused kernels, no gradient tracking).

FastLanguageModel.for_inference(model)

# Build a test prompt in Llama 3 format (no assistant content yet)
test_prompt = (
    f"{LLAMA3_BEGIN_OF_TEXT}"
    f"{LLAMA3_START_HEADER}system{LLAMA3_END_HEADER}\n\n"
    f"{SYSTEM_PROMPT}{LLAMA3_END_OF_TURN}"
    f"{LLAMA3_START_HEADER}user{LLAMA3_END_HEADER}\n\n"
    f"Convert this P2 INC request for team SEC into the company JSON schema."
    f"{LLAMA3_END_OF_TURN}"
    f"{LLAMA3_START_HEADER}assistant{LLAMA3_END_HEADER}\n\n"
)

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with __import__("torch").no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens          = 150,
        use_cache               = True,
        temperature             = 0.1,
        do_sample               = True,
        repetition_penalty      = 1.1,
        pad_token_id            = tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens
new_ids      = output_ids[0][inputs["input_ids"].shape[-1]:]
raw_output   = tokenizer.decode(new_ids, skip_special_tokens=False)
clean_output = raw_output.split(LLAMA3_END_OF_TURN)[0].strip()

print("Raw model output:")
print(repr(raw_output))
print("\nClean output (stripped at <|eot_id|>):")
print(clean_output)


In [ ]:
# ── Phase 3: Grade the inference output ──────────────────────────────────
# This closes the loop: the grader from Phase 2 scores what the trained model produces.

score = grade_json_output(clean_output)
label = "VALID JSON" if score == 1.0 else "INVALID JSON"

print(f"Grader score: {score:+.1f}  ->  {label}")

if score == 1.0:
    parsed = json.loads(clean_output)
    print("\nParsed keys:", list(parsed.keys()))
    for k, v in parsed.items():
        print(f"  {k}: {v}")
else:
    print("\nOutput did not parse as JSON.")
    print("This can happen after only 60 training steps on a 1B model.")
    print("Increase MAX_STEPS or use Llama-3.2-3B for better results.")


---
### Phase 3 Q&A — LoRA Rank Trade-offs: r=8 vs r=16

Each LoRA adapter for a weight matrix `W` (shape `[d_out, d_in]`) decomposes the update as:

$$\Delta W = B \cdot A, \quad B \in \mathbb{R}^{d_{out} \times r},\; A \in \mathbb{R}^{r \times d_{in}}$$

The rank `r` controls how many parameters are added:

| Property | r = 8 | r = 16 |
|----------|-------|--------|
| Parameters per adapter | `(d_out + d_in) × 8` | `(d_out + d_in) × 16` |
| Total trainable params | ~baseline | ~2x baseline |
| VRAM (training overhead) | lower | ~1.5x higher |
| Representational capacity | lower | higher |
| Risk of overfitting (small dataset) | lower | higher |
| Training speed | faster | ~5–10% slower |

**Recommendation for this exercise:**  
- `r=8` is the right choice — the dataset has only 50 rows, so higher rank gains nothing and risks overfitting.  
- `r=16` makes sense when: (a) dataset > 1000 rows, (b) the task is complex, (c) VRAM budget allows.


In [ ]:
# ── Phase 3: Parameter count comparison r=8 vs r=16 ──────────────────────
# Demonstrates the math without re-training.

# Llama-3.2-1B approximate projection dimensions
HIDDEN_DIM = 2048
HEAD_DIM   = 64
N_HEADS    = 32
FFN_INTER  = 8192
N_LAYERS   = 16

def lora_params_per_layer(r: int) -> dict:
    """Count LoRA adapter parameters per transformer layer."""
    # attention: q, k, v, o projections
    q = (HIDDEN_DIM + HIDDEN_DIM) * r         # [d_model -> d_model]
    k = (HIDDEN_DIM + HIDDEN_DIM) * r
    v = (HIDDEN_DIM + HIDDEN_DIM) * r
    o = (HIDDEN_DIM + HIDDEN_DIM) * r
    # feed-forward: gate, up, down
    gate = (HIDDEN_DIM + FFN_INTER) * r
    up   = (HIDDEN_DIM + FFN_INTER) * r
    down = (FFN_INTER  + HIDDEN_DIM) * r
    return {"q": q, "k": k, "v": v, "o": o,
            "gate": gate, "up": up, "down": down,
            "total": q+k+v+o+gate+up+down}

for r in [8, 16]:
    per_layer = lora_params_per_layer(r)
    total     = per_layer["total"] * N_LAYERS
    print(f"\n--- LoRA rank r={r} ---")
    for k, v in per_layer.items():
        if k != "total":
            print(f"  {k:6s} projection: {v:>10,} params")
    print(f"  Per-layer total:   {per_layer['total']:>10,} params")
    print(f"  All {N_LAYERS} layers total: {total:>10,} params")

# Ratio
r8  = lora_params_per_layer(8)["total"]  * N_LAYERS
r16 = lora_params_per_layer(16)["total"] * N_LAYERS
print(f"\nr=16 / r=8 ratio: {r16/r8:.2f}x more parameters")


---
## Phase 3 (No-GPU) — API Inference + Grader Integration

The cells below run **entirely on CPU** — no GPU or Unsloth required.

| Cell | What it does | GPU needed? |
|------|-------------|-------------|
| Gemini install | `pip install google-generativeai` | No |
| API inference demo | Call Gemini 2.5 Flash Lite via Google AI | No |
| Grader integration | Score outputs with Phase 2 grader | No |
| Standalone Modelfile | Write Ollama config template | No |

> **Why Gemini API?**  `gemini-2.5-flash-lite` is fast and free-tier accessible.  
> This demonstrates the full inference + grading pipeline without a local GPU.  
> After GPU training, swap the API calls for `ollama run company-syntax` and the pipeline is identical.

In [ ]:
%%capture
# Install Google Generative AI client — CPU-only, no CUDA dependency
!pip install google-generativeai

In [ ]:
# ── Phase 3 (No-GPU): Gemini API inference demo ───────────────────────────
# Uses the same SYSTEM_PROMPT and input format defined in Phase 1.
# Demonstrates the inference half of the LoRA fine-tuning pipeline
# without needing a local GPU.
#
# Setup:
#   1. Get a free API key at https://aistudio.google.com/apikey
#   2. In Colab: click the 🔑 (Secrets) icon → add GEMINI_API_KEY
#   3. Re-run this cell

import os
import google.generativeai as genai

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
GEMINI_MODEL   = "gemini-2.5-flash-lite"

# These prompts match the template patterns used in Phase 1 training data
TEST_PROMPTS = [
    "Convert this P1 INC request for team SEC into the company JSON schema.",
    "Normalize acronym RCA for department OPS with P2 priority and return strict JSON only.",
    "Format the internal NET CAPA workflow as the approved response object. Severity is P3.",
    "Translate company shorthand IAM for team HR. Use the standard schema and mark priority as P1.",
    "Create the final machine-readable company record for FIN / CRQ / P2. Return JSON only.",
]

gemini_outputs = []   # populated below if key is present

if not GEMINI_API_KEY:
    print("GEMINI_API_KEY not set — skipping live inference.")
    print("Add key via Colab Secrets (🔑 icon) and re-run.\n")
    # Provide realistic mock outputs so the downstream grader cell still works
    gemini_outputs = [
        {"prompt": p, "output": json.dumps({
            "request_code":      code,
            "request_expansion": exp,
            "request_type":      rtype,
            "department":        dept,
            "priority":          prio,
        }, separators=(",", ":"))}
        for p, (code, exp, rtype, dept, prio) in zip(TEST_PROMPTS, [
            ("INC",  "incident",                            "service_incident",       "SEC", "P1"),
            ("RCA",  "root_cause_analysis",                 "post_incident_review",   "OPS", "P2"),
            ("CAPA", "corrective_action_preventive_action", "corrective_action_plan", "NET", "P3"),
            ("IAM",  "identity_and_access_management",      "access_request",         "HR",  "P1"),
            ("CRQ",  "change_request",                      "planned_change",         "FIN", "P2"),
        ])
    ]
    print("Using mock outputs for demonstration.\n")
else:
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(
        model_name        = GEMINI_MODEL,
        system_instruction= SYSTEM_PROMPT,
    )
    gen_config = genai.GenerationConfig(
        max_output_tokens = 150,
        temperature       = 0.1,
    )
    print(f"Calling {GEMINI_MODEL} for {len(TEST_PROMPTS)} prompts...\n")
    for prompt in TEST_PROMPTS:
        response = model.generate_content(prompt, generation_config=gen_config)
        output   = response.text.strip()
        gemini_outputs.append({"prompt": prompt, "output": output})

print(f"{'#':>2}  Prompt (truncated to 55 chars)")
print(f"{'':>4}  Output (truncated to 100 chars)")
print("-" * 70)
for i, item in enumerate(gemini_outputs, 1):
    print(f"{i:>2}  {item['prompt'][:55]}")
    print(f"    {item['output'][:100]}")
    print()

In [ ]:
# ── Phase 3 (No-GPU): Grade Gemini outputs with the Phase 2 grader ───────
# Closes the loop: Phase 2 grader scores the inference outputs from above.
# This is exactly what an RFT training loop does after each model rollout.

scores      = [grade_json_output(item["output"]) for item in gemini_outputs]
valid_count = sum(1 for s in scores if s == 1.0)

print(f"{'#':>2}  {'Score':>6}  {'Result':12}  Prompt (truncated)")
print("-" * 72)
for i, (item, score) in enumerate(zip(gemini_outputs, scores), 1):
    label = "VALID JSON" if score == 1.0 else "INVALID JSON"
    print(f"{i:>2}  {score:+6.1f}  {label:12}  {item['prompt'][:45]}...")

avg = sum(scores) / len(scores)
print("-" * 72)
print(f"\nValid JSON : {valid_count}/{len(scores)}")
print(f"Avg reward : {avg:+.2f}")
print(f"\nHow this feeds into RFT:")
print("  PPO  — each score is a per-rollout scalar reward fed to the advantage estimator.")
print("  DPO  — valid output becomes y_w (chosen), invalid becomes y_l (rejected).")

# Show parsed keys for first valid output
for item, score in zip(gemini_outputs, scores):
    if score == 1.0:
        parsed = json.loads(item["output"])
        print(f"\nParsed keys from first valid output: {list(parsed.keys())}")
        break

In [ ]:
# ── Phase 3 (No-GPU): Standalone Ollama Modelfile ────────────────────────
# Generates the Ollama deployment config without requiring GPU training.
# After GPU training produces a GGUF file, point FROM at it and run ollama create.

GGUF_QUANT    = "q4_k_m"
gguf_filename = f"model-{GGUF_QUANT.upper()}.gguf"

modelfile_text = f"""FROM ./{gguf_filename}

SYSTEM \"\"\"
{SYSTEM_PROMPT}
\"\"\"

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|eot_id|>"
"""

GGUF_DIR.mkdir(parents=True, exist_ok=True)
modelfile_path = GGUF_DIR / "Modelfile"
modelfile_path.write_text(modelfile_text, encoding="utf-8")

print(f"Modelfile written: {modelfile_path}")
print("\nContents:")
print("-" * 50)
print(modelfile_text)
print("-" * 50)
print("\nDeploy after GPU training:")
print(f"  ollama create company-syntax -f {modelfile_path}")
print("  ollama run company-syntax")
print("\nAPI call equivalent (once Ollama is running):")
print('  curl http://localhost:11434/api/generate \\')
print('    -d \'{"model":"company-syntax",')
print('         "prompt":"Convert this P1 INC request for team SEC.",')
print('         "stream":false}\'')

In [ ]:
# ── Phase 3: Merge LoRA adapter into base model ───────────────────────────
# After merging, the result is a standalone model with no adapter dependency.
# Saves as 16-bit (BF16/FP16) for maximum compatibility.

print("Merging LoRA adapter into base model (16-bit)...")
model.save_pretrained_merged(
    str(MERGED_DIR),
    tokenizer,
    save_method = "merged_16bit",
)

merged_files = list(MERGED_DIR.glob("*"))
merged_size  = sum(f.stat().st_size for f in merged_files if f.is_file()) / (1024**2)
print(f"\nMerged model saved to: {MERGED_DIR}")
print(f"Total size: {merged_size:.1f} MB ({len(merged_files)} files)")


In [ ]:
# ── Phase 3: Export as GGUF (q4_k_m) ─────────────────────────────────────
# q4_k_m = 4-bit quantization, K-quants medium variant.
# Typically reduces a 1B 16-bit model (~2 GB) to ~700 MB.
# Compatible with llama.cpp and Ollama.

print("Exporting GGUF (q4_k_m) — this may take 1-3 minutes...")
GGUF_QUANT = "q4_k_m"

model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method = GGUF_QUANT,
)

gguf_files = list(GGUF_DIR.glob("*.gguf"))
print(f"\nGGUF files exported to: {GGUF_DIR}")
for f in gguf_files:
    print(f"  {f.name}  ({f.stat().st_size/(1024**2):.1f} MB)")


In [ ]:
# ── Phase 3: Generate Ollama Modelfile ────────────────────────────────────
# This file tells Ollama how to run the GGUF model.

gguf_filename  = f"model-{GGUF_QUANT.upper()}.gguf"
modelfile_text = f"""FROM ./{gguf_filename}

SYSTEM \"\"\"
{SYSTEM_PROMPT}
\"\"\"

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|eot_id|>"
"""

modelfile_path = GGUF_DIR / "Modelfile"
modelfile_path.write_text(modelfile_text, encoding="utf-8")

print(f"Ollama Modelfile written: {modelfile_path}")
print("\nModelfile contents:")
print("-" * 50)
print(modelfile_text)
print("-" * 50)
print("\nTo run with Ollama:")
print(f"  ollama create company-syntax -f {modelfile_path}")
print("  ollama run company-syntax")


---
### Phase 3 Q&A — Deployment: GGUF + Ollama

#### Why GGUF?
GGUF (GPT-Generalized Unified Format) is the standard format for **llama.cpp** and **Ollama**.  
It packages weights + tokenizer + metadata into a single portable file.

#### What `q4_k_m` means
| Part | Meaning |
|------|---------|
| `q4` | 4-bit weight quantization |
| `k`  | K-quants method (better quality than naive Q4) |
| `m`  | Medium variant (balance of size and accuracy) |

For a 1B model, `q4_k_m` gives ~0.7 GB vs ~2 GB for FP16 — 3x smaller.

#### Local Deployment Workflow
```bash
# 1. Install Ollama (https://ollama.com)
# 2. Create the model from your exported GGUF
ollama create company-syntax -f ./outputs/gguf_model/Modelfile

# 3. Run inference
ollama run company-syntax
>>> Convert this P1 IAM request for team SEC into the company JSON schema.

# 4. Or via API
curl http://localhost:11434/api/generate -d '{
  "model": "company-syntax",
  "prompt": "Convert this P1 IAM request for team SEC into the company JSON schema.",
  "stream": false
}'
```

The `stop: "<|eot_id|>"` parameter in the Modelfile ensures Ollama stops cleanly  
at the end-of-turn token — the same boundary we trained the model to produce.


---
## Save & Download Outputs

Mount Google Drive to persist the adapter, merged model, and GGUF file.  
If you skip this, outputs are lost when the Colab runtime resets.


In [ ]:
# ── Save to Google Drive (optional) ──────────────────────────────────────
# Comment out if you don't need persistent storage.

try:
    from google.colab import drive
    drive.mount("/content/drive")

    import shutil
    dest = Path("/content/drive/MyDrive/KDU_2026_AI_Phase3_outputs")
    dest.mkdir(parents=True, exist_ok=True)

    # Copy outputs (adapter, merged, gguf, dataset)
    shutil.copytree("outputs", str(dest / "outputs"), dirs_exist_ok=True)
    shutil.copy(str(DATA_PATH), str(dest / DATA_PATH.name))

    print(f"Outputs saved to Google Drive: {dest}")
    for item in dest.rglob("*"):
        if item.is_file():
            print(f"  {item.relative_to(dest)}  ({item.stat().st_size/1024:.0f} KB)")
except Exception as e:
    print(f"Drive save skipped: {e}")
    print("Use the Files panel on the left to download outputs manually.")


In [ ]:
# ── Zip and download (alternative to Drive) ───────────────────────────────
# Creates a single zip of all outputs for direct browser download.

import shutil
zip_path = shutil.make_archive("phase3_outputs", "zip", "outputs")
print(f"Created: {zip_path}  ({Path(zip_path).stat().st_size/(1024**2):.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
    print("Download started.")
except Exception:
    print("Not in Colab — find phase3_outputs.zip in the file browser.")


---
## Summary

### What was built

| Deliverable | Location | GPU needed? |
|-------------|----------|-------------|
| Training dataset | `data/company_syntax_train.jsonl` | No |
| JSON grader | `grade_json_output()` (this notebook) | No |
| LoRA rank analysis | `lora_params_per_layer()` (this notebook) | No |
| Gemini inference demo | `gemini_outputs` (this notebook) | No |
| Ollama Modelfile | `outputs/gguf_model/Modelfile` | No |
| LoRA adapter | `outputs/lora_adapter/` | **Yes (T4 GPU)** |
| Merged model | `outputs/merged_model/` | **Yes (T4 GPU)** |
| GGUF file | `outputs/gguf_model/*.gguf` | **Yes (T4 GPU)** |

### No-GPU execution order

Run these cells top-to-bottom without enabling GPU:

1. **Setup** — `code-base-imports`
2. **Phase 1** — `code-p1-constants` → `code-p1-functions` → `code-p1-generate` → `code-p1-preview`
3. **Phase 2** — `code-p2-grader` → `code-p2-test`
4. **Phase 3** — `code-p3-config` → `code-p3-rank-comparison` → install Gemini → Gemini inference demo → grader integration → standalone Modelfile

### Key Takeaways

| Topic | Finding |
|-------|---------|
| `<|eot_id|>` | Missing EOT causes runaway generation and broken JSON at inference |
| LoRA rank r=8 | ~1–2% trainable params — sufficient for a 50-row structured-output task |
| r=8 vs r=16 | r=16 doubles adapter parameters with no benefit on small datasets |
| Grader reward | Binary +1/−1 cleanly separates valid/invalid JSON; integrates directly into PPO/DPO |
| GGUF q4_k_m | 3× size reduction vs FP16 with minimal quality loss; runs on CPU/GPU with Ollama |
| Gemini API | `gemini-2.5-flash-lite` inference — demonstrates the full deployment loop without local GPU |

### Report Checklist

- [x] Formatting issues explained (`<|eot_id|>` trap + Llama 3 format fix)
- [x] RFT reward logic (grader function + PPO/DPO integration)
- [x] LoRA parameter trade-offs (r=8 vs r=16 with math)
- [x] Deployment approach (GGUF / Ollama workflow + Modelfile)
- [x] API inference demo (Gemini 2.5 Flash Lite, no GPU required)